# Summary SimFin

Health check + coverage report for SimFin data in the local DuckDB.

SimFin writes four tables: **`income`**, **`balance`**, **`cashflow`** (financial statements) and **`companies`** (metadata).

Row-level retrieval goes through `irp.data.simfin.fundamentals()` / `companies()`. Aggregations stay in SQL via the shared `db()` connection.

In [ ]:
import pandas as pd
from IPython.display import display

from irp.data._common import db
from irp.data.simfin import companies as _companies
from irp.data.simfin import fundamentals

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [ ]:
_META = {
    'Ticker',
    'SrcId',
    'Src',
    'Currency',
    'Fiscal Year',
    'Fiscal Period',
    'Report Date',
    'Publish Date',
    'Restated Date',
    'Market',
    'Period',
}


def summarize(df: pd.DataFrame, name: str) -> None:
    tickers = sorted(df['Ticker'].unique())
    items = [c for c in df.columns if c not in _META]
    print(f'=== {name} ===')
    print(f'Rows: {len(df):,}  |  Tickers: {len(tickers)}')
    print(
        f'Fiscal Year : {int(df["Fiscal Year"].min())} – {int(df["Fiscal Year"].max())}'
    )
    print(
        f'Report Date : {df["Report Date"].min().date()} – {df["Report Date"].max().date()}'
    )
    print(f'\nTickers ({len(tickers)}): {", ".join(tickers)}')
    print(f'\nStatement items ({len(items)}):')
    for col in items:
        n = df[col].notna().sum()
        print(f'  {col:<50s}  {n:>6,}  ({100 * n / len(df):.0f}%)')

## Tables

Schema, row count, and key stats for each table written by `irp.sources.sim_fin.SimFinSource`.

### `income` / `balance` / `cashflow`

One row per (Ticker, Fiscal Year, Fiscal Period, Period). `Period` is `'A'` (annual) or `'Q'` (quarterly).

| Column | Type | Meaning |
|---|---|---|
| Ticker | VARCHAR | Uppercase symbol |
| Fiscal Year | BIGINT | Calendar year the fiscal year ends |
| Fiscal Period | VARCHAR | `FY`, `Q1`–`Q4` |
| Period | VARCHAR | `A` or `Q` |
| Report Date | DATE | Date financials were reported |
| Currency | VARCHAR | Reporting currency (e.g. `USD`) |
| SrcId | VARCHAR | SimFin ticker ID |
| Src | VARCHAR | Always `simfin` |
| *(line items)* | DOUBLE | Statement-specific financial items |

In [ ]:
income_A = fundamentals(statement='income', variant='A')
income_Q = fundamentals(statement='income', variant='Q')
income = pd.concat([income_A, income_Q])

In [ ]:
summarize(income, 'Income Statement')

### Balance

In [ ]:
balance_A = fundamentals(statement='balance', variant='A')
balance_Q = fundamentals(statement='balance', variant='Q')
balance = pd.concat([balance_A, balance_Q])

In [ ]:
summarize(balance, 'Balance Sheet')

### Cashflow

In [ ]:
cashflow_A = fundamentals(statement='cashflow', variant='A')
cashflow_Q = fundamentals(statement='cashflow', variant='Q')
cashflow = pd.concat([cashflow_A, cashflow_Q])

In [ ]:
summarize(cashflow, 'Cash Flow Statement')

### `companies`

One row per Ticker. Company metadata used for enrichment and universe filtering.

| Column | Type | Meaning |
|---|---|---|
| Ticker | VARCHAR | Uppercase symbol |
| Company Name | VARCHAR | Full company name |
| Sector / Industry | VARCHAR | GICS sector and industry |
| Market | VARCHAR | Exchange market |
| ISIN | VARCHAR | International security identifier |
| SrcId / Src | VARCHAR | SimFin ID / loader name |

In [ ]:
_comp = _companies()

In [ ]:
_comp_cols = [c for c in _comp.columns if c not in {'Ticker', 'SrcId', 'Market'}]

print(f'Rows: {len(_comp):,}  |  Tickers: {_comp["Ticker"].nunique():,}')
print(f'Markets: {_comp["Market"].value_counts().to_dict()}')
print(f'Sectors: {_comp["Sector"].nunique()}  |  Industries: {_comp["Industry"].nunique()}')
print()
print('Market breakdown:')
display(
    _comp.groupby('Market')
    .agg(tickers=('Ticker', 'count'), sectors=('Sector', 'nunique'), industries=('Industry', 'nunique'))
    .reset_index()
)
print()
print(f'Column coverage ({len(_comp_cols)} columns):')
for col in _comp_cols:
    n = _comp[col].notna().sum()
    print(f'  {col:<40s}  {n:>5,}  ({100 * n / len(_comp):.0f}%)')

## Cross-table coverage

In [ ]:
display(db().execute("""
    SELECT
        COUNT(DISTINCT i.Ticker)                                             AS income_tickers,
        COUNT(DISTINCT b.Ticker)                                             AS balance_tickers,
        COUNT(DISTINCT cf.Ticker)                                            AS cashflow_tickers,
        COUNT(DISTINCT co.Ticker)                                            AS company_tickers,
        COUNT(DISTINCT i.Ticker) FILTER (WHERE co.Ticker IS NULL)            AS income_no_company,
        COUNT(DISTINCT co.Ticker) FILTER (WHERE i.Ticker IS NULL)            AS company_no_fundamentals
    FROM income i
    FULL OUTER JOIN balance   b  ON i.Ticker = b.Ticker
    FULL OUTER JOIN cashflow  cf ON i.Ticker = cf.Ticker
    FULL OUTER JOIN companies co ON i.Ticker = co.Ticker
""").df().T.rename(columns={0: 'count'}))

## Freshness — when was data last updated?

Latest `Report Date` per statement (annual and quarterly separately).

In [ ]:
display(db().execute("""
    SELECT 'income'   AS stmt, 'A' AS variant, MAX("Report Date") AS latest_report, COUNT(*) AS rows, COUNT(DISTINCT Ticker) AS tickers FROM income   WHERE Period = 'A'
    UNION ALL
    SELECT 'income',   'Q', MAX("Report Date"), COUNT(*), COUNT(DISTINCT Ticker) FROM income   WHERE Period = 'Q'
    UNION ALL
    SELECT 'balance',  'A', MAX("Report Date"), COUNT(*), COUNT(DISTINCT Ticker) FROM balance  WHERE Period = 'A'
    UNION ALL
    SELECT 'balance',  'Q', MAX("Report Date"), COUNT(*), COUNT(DISTINCT Ticker) FROM balance  WHERE Period = 'Q'
    UNION ALL
    SELECT 'cashflow', 'A', MAX("Report Date"), COUNT(*), COUNT(DISTINCT Ticker) FROM cashflow WHERE Period = 'A'
    UNION ALL
    SELECT 'cashflow', 'Q', MAX("Report Date"), COUNT(*), COUNT(DISTINCT Ticker) FROM cashflow WHERE Period = 'Q'
    ORDER BY stmt, variant
""").df())

## Problem Tickers

### Duplicate Tickers

In [ ]:
_dups = _comp[_comp.duplicated('Ticker', keep=False)].sort_values('Ticker')
print(f'Duplicate tickers: {_dups["Ticker"].nunique()}')
if len(_dups):
    display(_dups[['Ticker', 'Company Name', 'Market', 'SrcId']])
else:
    print('None.')

### Tickers Without Fundamentals

In [ ]:
_fund_tickers = set(income['Ticker']) | set(balance['Ticker']) | set(cashflow['Ticker'])
_no_fund = sorted(set(_comp['Ticker'].dropna()) - _fund_tickers)
print(f'Tickers with no fundamentals: {len(_no_fund)}')
print(_no_fund)

### Tickers Without Prices

In [ ]:
_price_tickers = set(
    db().execute('SELECT DISTINCT Ticker FROM prices').fetchdf()['Ticker']
)
_no_prices = sorted(set(_comp['Ticker'].dropna()) - _price_tickers)
print(f'Tickers with no prices: {len(_no_prices)}')
print(_no_prices)

### Missing Statements by Period

In [ ]:
_stmt_map = {'income': income, 'balance': balance, 'cashflow': cashflow}

In [ ]:
def _fp(df):
    # annual Fiscal Period varies by statement ('FY' for income/cashflow, 'Q4' for balance)
    # normalize all annual rows to 'FY' so the pivot key is consistent across statements
    fp = df['Period'].map({'A': 'FY'}).fillna(df['Fiscal Period'])
    return df[['Ticker', 'Fiscal Year']].assign(**{'Fiscal Period': fp})


all_combos = pd.concat(
    [_fp(df).assign(stmt=name) for name, df in _stmt_map.items()]
).drop_duplicates()

pivot = (
    all_combos.assign(present=1)
    .pivot_table(
        index=['Ticker', 'Fiscal Year', 'Fiscal Period'],
        columns='stmt',
        values='present',
    )
    .fillna(0)
    .reset_index()
)
pivot.columns.name = None

stmts = list(_stmt_map.keys())
problems = pivot[pivot[stmts].eq(0).any(axis=1)].copy()
problems = problems.sort_values(['Ticker', 'Fiscal Year', 'Fiscal Period']).reset_index(
    drop=True
)
for col in stmts:
    problems[col] = problems[col].map({1: '✓', 0: '✗'})

print(
    f'Problem periods: {len(problems):,}  |  Tickers affected: {problems["Ticker"].nunique()}'
)
display(problems)

In [ ]:
problem_tickers = problems['Ticker'].unique()
problem_tickers

### Filings per Ticker

In [ ]:
_q = income[income['Period'] == 'Q']

_filings = income.groupby('Ticker').agg(
    total=('Period', 'count'),
    annual=('Period', lambda s: (s == 'A').sum()),
    quarterly=('Period', lambda s: (s == 'Q').sum()),
)
for _qtr in ('Q1', 'Q2', 'Q3', 'Q4'):
    _filings[_qtr] = _q[_q['Fiscal Period'] == _qtr].groupby('Ticker').size()
_filings = _filings.fillna(0).astype(int).reset_index()

display(_filings)

In [ ]:
_filings['total'].hist(bins=25)

### Tickers Without Shares

In [ ]:
_no_shares = (
    income_A.groupby('Ticker')[['Shares (Basic)', 'Shares (Diluted)']]
    .apply(lambda g: g.isna().all())
    .all(axis=1)
)
_no_shares_tickers = _no_shares[_no_shares].index.tolist()
print(f'Tickers with no shares data (annual): {len(_no_shares_tickers)}')
print(_no_shares_tickers)

### Duplicate Keys

In [ ]:
_key = ['Ticker', 'Fiscal Year', 'Fiscal Period', 'Period']
for _name, _df in _stmt_map.items():
    _dups = _df[_df.duplicated(subset=_key, keep=False)]
    print(
        f'{_name}: {len(_dups)} duplicate rows across {_dups["Ticker"].nunique()} tickers'
    )
    if len(_dups):
        display(_dups[_key].sort_values(_key))

### Annual Coverage Gaps

In [ ]:
_gaps = []
for _ticker, _g in income_A.groupby('Ticker'):
    _years = sorted(_g['Fiscal Year'].unique())
    _missing = sorted(set(range(_years[0], _years[-1] + 1)) - set(_years))
    if _missing:
        _gaps.append(
            {'Ticker': _ticker, 'Missing Years': _missing, 'Count': len(_missing)}
        )

if _gaps:
    _gaps_df = (
        pd.DataFrame(_gaps).sort_values('Count', ascending=False).reset_index(drop=True)
    )
    print(f'Tickers with annual gaps: {len(_gaps_df)}')
    display(_gaps_df)
else:
    print('No annual coverage gaps found.')

### Quarterly Coverage Gaps

In [ ]:
_expected_qtrs = {'Q1', 'Q2', 'Q3', 'Q4'}
_q_map = {'income': income_Q, 'balance': balance_Q, 'cashflow': cashflow_Q}

for _name, _df in _q_map.items():
    _fy_min = _df.groupby('Ticker')['Fiscal Year'].min().to_dict()
    _fy_max = _df.groupby('Ticker')['Fiscal Year'].max().to_dict()
    _gaps_q = []
    for (_ticker, _fy), _g in _df.groupby(['Ticker', 'Fiscal Year']):
        if _fy == _fy_min[_ticker] or _fy == _fy_max[_ticker]:
            continue
        _missing = sorted(_expected_qtrs - set(_g['Fiscal Period'].unique()))
        if _missing:
            _gaps_q.append({'Ticker': _ticker, 'Fiscal Year': int(_fy), 'Missing': _missing, 'Count': len(_missing)})

    if _gaps_q:
        _gaps_q_df = (
            pd.DataFrame(_gaps_q)
            .sort_values(['Count', 'Ticker', 'Fiscal Year'], ascending=[False, True, True])
            .reset_index(drop=True)
        )
        print(f'{_name}: {len(_gaps_q_df)} ticker/year combos with gaps  |  {_gaps_q_df["Ticker"].nunique()} tickers affected')
        display(_gaps_q_df)
    else:
        print(f'{_name}: no quarterly coverage gaps')

### Tickers Not in Companies Table

In [ ]:
_comp_tickers = set(_companies()['Ticker'])
_fund_tickers = set(income['Ticker'])
_no_meta = sorted(_fund_tickers - _comp_tickers)
print(f'Tickers in fundamentals but not in companies: {len(_no_meta)}')
print(_no_meta)

### Currency Changes

In [ ]:
_currency_changes = [
    {'Ticker': _t, 'Currencies': sorted(_g['Currency'].unique())}
    for _t, _g in income.groupby('Ticker')
    if _g['Currency'].nunique() > 1
]
_curr_df = pd.DataFrame(_currency_changes)
print(f'Tickers with currency changes: {len(_curr_df)}')
display(_curr_df)